In [1]:
import pandas as pd

df = pd.read_csv("glass.csv")

print("Shape of dataset:", df.shape)
print("Columns:", df.columns)

display(df.head())

Shape of dataset: (214, 10)
Columns: Index(['RI', 'Na', 'Mg', 'Al', 'Si', 'K', 'Ca', 'Ba', 'Fe', 'Type'], dtype='object')


,RI,Na,Mg,Al,Si,K,Ca,Ba,Fe,Type
0,1.52101,13.64,4.49,1.10,71.78,0.06,8.75,0.0,0.0,1
1,1.51761,13.89,3.60,1.36,72.73,0.48,7.83,0.0,0.0,1
2,1.51618,13.53,3.55,1.54,72.99,0.39,7.78,0.0,0.0,1
3,1.51766,13.21,3.69,1.29,72.61,0.57,8.22,0.0,0.0,1
4,1.51742,13.27,3.62,1.24,73.08,0.55,8.07,0.0,0.0,1


In [2]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

df["y"] = (df["Type"] == 1).astype(int)

columns_to_drop = ["Type", "y"]
if "Id_column_name" in df.columns:
    columns_to_drop.append("Id_column_name")

X = df.drop(columns=columns_to_drop).values
y = df["y"].values

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

print("Data preprocessed, split, and scaled successfully!")

Data preprocessed, split, and scaled successfully!


In [3]:
import numpy as np

def sigmoid(z):
    return 1 / (1 + np.exp(-z))

def predict_proba(X, w, b):
    z = X @ w + b
    p = sigmoid(z)
    return p

def loss(y, p):
    epsilon = 1e-15
    p = np.clip(p, epsilon, 1 - epsilon)
    return -np.mean(y * np.log(p) + (1 - y) * np.log(1 - p))

def update_weights(X, y, w, b, lr):
    p = predict_proba(X, w, b)
    error = p - y
    w = w - lr * (X.T @ error) / len(y)
    b = b - lr * np.mean(error)
    return w, b

In [4]:
from tqdm import tqdm
import numpy as np

w = np.zeros(X_train.shape[1])
b = 0.0
lr = 0.1
epochs = 100

print("Starting training...")

progress_bar = tqdm(range(epochs), desc="Training")

for epoch in progress_bar:
    w, b = update_weights(X_train, y_train, w, b, lr)

    current_p = predict_proba(X_train, w, b)
    current_loss = loss(y_train, current_p)

    progress_bar.set_postfix(loss=f"{current_loss:.4f}")

def predict_label(p, threshold=0.5):
    return (p >= threshold).astype(int)

test_probs = predict_proba(X_test, w, b)

preds_05 = predict_label(test_probs, threshold=0.5)
preds_07 = predict_label(test_probs, threshold=0.7)

print("\nPredictions with 0.5 threshold:", preds_05[:10])
print("Predictions with 0.7 threshold:", preds_07[:10])

Starting training...


Training: 100%|██████████| 100/100 [00:00<00:00, 307.99it/s, loss=0.4978]


Predictions with 0.5 threshold: [0 0 1 0 0 0 1 0 0 0]
Predictions with 0.7 threshold: [0 0 0 0 0 0 0 0 0 0]
